## BPE : considering only GPT-2

In [ ]:
text = """ After getting this base vocabulary, we add new tokens until the desired vocabulary size is reached by learning merges, which are rules to merge two elements of the existing vocabulary together into a new one. So, at the beginning these merges will create tokens with two characters, and then, as training progresses, longer subwords.

At any step during the tokenizer training, the BPE algorithm will search for the most frequent pair of existing tokens (by “pair,” here we mean two consecutive tokens in a word). That most frequent pair is the one that will be merged, and we rinse and repeat for the next step. """

In [ ]:
ord("!") # this is for only character level encoding

In [ ]:
[ord(cr) for cr in text]

In [ ]:
tokens = text.encode("utf-8")
tokens = list(map(int,tokens))
print("-----")
print(text)
print("----")
print(tokens)

In [ ]:
## creating byte pairs
def get_stats(ids):
  counts = {}
  for pair in zip(ids,ids[1:]):
    counts[pair] = counts.get(pair,0) + 1
  return counts


In [ ]:
stats = get_stats(tokens)
print(stats)
print(sorted({(v,k) for k,v in stats.items()},reverse=True)) # taking hisghest occureance front

In [ ]:
chr(32),chr(116)

In [ ]:
top_pair = max(stats,key=stats.get)
top_pair

In [ ]:
def merge(ids,pair,idx):
  """
  ids : tokens
  pair : top pair
  idx :  the next top pair value
  """
  new_ids = [] # new tokens
  i = 0
  while i < len(ids):
    if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
      new_ids.append(idx)
      i +=2
    else:
      new_ids.append(ids[i])
      i += 1
  return new_ids

In [ ]:
merge([22,32,22,32,4],(22,32),256) # example

In [ ]:
## example : for original tokens
merge(tokens,top_pair,256)

In [ ]:
vocab_size = 300 # that much size i want and it is hyperparameter
num_merges = vocab_size  - 256
ids = tokens # already in list

merges = {} # required for decoding
for i in range(num_merges):
  stats = get_stats(ids)
  pair = max(stats,key=stats.get)
  idx = 256 + i
  print(f"merging {pair} into new token {idx}")
  ids = merge(ids,pair,idx)
  merges[pair] = idx

In [ ]:
len(tokens)/len(ids) # how much comperassion

In [ ]:
## merges dict
merges # (int,int) -> int

In [ ]:
## decoder in tokenizer
## given token sequence let's go back and create the unicode text
vocab = {idx:bytes([idx]) for idx in range(256)}
for (p0,p1),i in merges.items():
  vocab[i] = vocab[p0] + vocab[p1]
vocab

def decode(ids):
  """
  ids :  tokens
  return decoded string"""
  tokens = b"".join(vocab[idx] for idx in ids)
  text = tokens.decode("utf-8",errors="replace")
  return text

In [ ]:
print(decode([128])) # replace ment syabol : ?

In [ ]:
### encode
### string -> tokens
def encode(text):
  tokens = list(map(int,text.encode("utf-8")))
  while len(tokens) >= 2:
    stats = get_stats(tokens)
    pair = min(stats,key=lambda p : merges.get(p,float("inf")))
    if pair not in merges:
      break
    idx = merges[pair]
    tokens = merge(tokens,pair,idx)
  return tokens

In [ ]:
encode("hello world")

In [ ]:
import regex as re
gpt2pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""") # we are not trying to merge accross letters , puncations , symbols

In [ ]:
print(re.findall(gpt2pat,"""hello'123v world! how's"""))
## here individual level merging will happen : like "hello" will internally merges with itself only not with some spave in "d !"

#### Some special tokens

In [ ]:
!wget https://openaipublic.blob.core.windows.net/gpt-2/models/1558M/vocab.bpe
!wget https://openaipublic.blob.core.windows.net/gpt-2/models/1558M/encoder.json

In [ ]:
import os ,json
with open('encoder.json','r') as f:
  encoder = json.load(f)

with open("vocab.bpe","r",encoding="utf-8") as f:
  bpe_data = f.read()
bpe_merges = [tuple(merge_str.split()) for merge_str in bpe_data.split("\n")[1:-1]]

In [ ]:
len(encoder)